# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [5]:
!pip install -q openai pydantic pypdf

import os
print("API_GATEWAY_KEY loaded:", os.getenv("API_GATEWAY_KEY") is not None)

from pypdf import PdfReader

file_path = "../05_src/documents/ManagingOneself_Drucker_HBR.pdf"

reader = PdfReader(file_path)

document_text = ""
for page in reader.pages:
    document_text += (page.extract_text() or "") + "\n"

print("Loaded document_text length:", len(document_text))
print(document_text[:800])

API_GATEWAY_KEY loaded: True
Loaded document_text length: 51478
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief— the core idea
The Idea in Practice— putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
 
B
 



## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
from openai import OpenAI
from pydantic import BaseModel, Field

# Gateway client (IMPORTANT)
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
)

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="One paragraph explaining relevance for an AI professional.")
    Summary: str = Field(description="Concise summary, max 1000 tokens.")
    Tone: str
    InputTokens: int
    OutputTokens: int

developer_prompt = """
You are a careful assistant that summarizes documents for AI professionals.

You must produce output strictly matching the provided JSON schema.

Rules:
- Use the exact tone specified by the user.
- Relevance must be one paragraph maximum.
- Summary must be concise and no longer than 1000 tokens.
- Do not hallucinate facts not present in the document.
"""

tone = "Bureaucratese"

user_prompt_template = """
Summarize the following document.

Tone requirement:
Write the summary in a clearly recognizable style of: {tone}

Document text:
{document_text}
"""

user_prompt = user_prompt_template.format(
    tone=tone,
    document_text=document_text
)

response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt},
    ],
    text_format=SummaryOutput
)

result = response.output_parsed
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

print(result.model_dump_json(indent=2))


{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "This seminal work discusses the imperative of self-management in the modern knowledge economy, emphasizing the need for individuals to take personal responsibility for their career trajectories. For AI professionals, understanding one's strengths, learning preferences, and ethical values is crucial for maximizing productivity and engagement, particularly in a field characterized by rapid change and innovation.",
  "Summary": "In \"Managing Oneself,\" Peter F. Drucker elucidates the necessity for individuals to adopt a proactive approach to self-management within today's knowledge economy, where success hinges on self-awareness and personal responsibility. Recognizing that organizations no longer manage careers, Drucker posits that one must act as their own CEO, strategically navigating their professional journey over potentially five decades. Key self-inquiry questions include identifying personal strength

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [9]:
import json

def evaluate_summary(document_text, summary_text, tone):
    prompt = f"""
You are an evaluation assistant.

Evaluate the SUMMARY based on the DOCUMENT.

You must output ONLY valid JSON with these exact keys:

SummarizationScore
SummarizationReason
CoherenceScore
CoherenceReason
TonalityScore
TonalityReason
SafetyScore
SafetyReason

Scoring:
- Scores must be floats between 0 and 1.
- Reasons must be 1 paragraph max each.

Tone requirement (for evaluation): neutral and analytic.

DOCUMENT:
{document_text}

SUMMARY:
{summary_text}

TARGET TONE (the summary was supposed to follow):
{tone}

Evaluation rules:
- SummarizationScore = coverage + factual consistency.
- CoherenceScore = clarity + logical flow + non-redundancy.
- TonalityScore = how strongly it matches the target tone.
- SafetyScore = no harmful, unsafe, discriminatory, or privacy-violating content.
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
        temperature=0
    )

    raw = response.output_text.strip()

    # Safety: if model returns extra text, try to recover JSON
    if not raw.startswith("{"):
        raw = raw[raw.find("{"):]
    if not raw.endswith("}"):
        raw = raw[:raw.rfind("}")+1]

    return json.loads(raw)

evaluation_result = evaluate_summary(
    document_text=document_text,
    summary_text=result.Summary,
    tone=tone
)

print(json.dumps(evaluation_result, indent=2))


{
  "SummarizationScore": 0.9,
  "SummarizationReason": "The summary effectively captures the core ideas of Drucker's article, including the importance of self-management, self-awareness, and the need for ethical alignment. It covers the key self-inquiry questions and the implications of misalignment between personal and organizational values, providing a comprehensive overview.",
  "CoherenceScore": 0.85,
  "CoherenceReason": "The summary presents a clear and logical flow of ideas, transitioning smoothly from one concept to another. However, some sentences could be more concise to enhance clarity and reduce redundancy.",
  "TonalityScore": 0.4,
  "TonalityReason": "The summary maintains a neutral and analytic tone but does not align with the bureaucratese tone, which typically features more formal and complex language. The language used is straightforward and lacks the formality expected in bureaucratese.",
  "SafetyScore": 1.0,
  "SafetyReason": "The content of the summary is safe an

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [11]:
class EnhancedSummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="One paragraph explaining relevance for an AI professional.")
    Summary: str = Field(description="Concise summary, max 1000 tokens.")
    Tone: str
    InputTokens: int
    OutputTokens: int

developer_prompt_enhance = """
You are a careful assistant that improves summaries for AI professionals.

You MUST:
- Preserve factual accuracy (no hallucinations).
- Keep the same tone requested by the user.
- Improve clarity, completeness, and precision.
- Remove redundancy.
- Output must strictly match the JSON schema.
"""

user_prompt_enhance_template = """
You are given:

1) The full document
2) The original summary
3) Evaluation scores and feedback

Task:
Rewrite the summary to improve it.

Tone requirement:
Write the summary in a clearly recognizable style of: {tone}

Constraints:
- Summary must be concise and no longer than 1000 tokens.
- Keep factual accuracy.
- Improve weaknesses mentioned in evaluation.

DOCUMENT:
{document_text}

ORIGINAL SUMMARY:
{original_summary}

EVALUATION FEEDBACK (JSON):
{evaluation_json}
"""

user_prompt_enhance = user_prompt_enhance_template.format(
    tone=tone,
    document_text=document_text,
    original_summary=result.Summary,
    evaluation_json=json.dumps(evaluation_result, indent=2)
)

enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_prompt_enhance},
        {"role": "user", "content": user_prompt_enhance},
    ],
    text_format=EnhancedSummaryOutput
)

enhanced_result = enhanced_response.output_parsed
enhanced_result.InputTokens = enhanced_response.usage.input_tokens
enhanced_result.OutputTokens = enhanced_response.usage.output_tokens

print(enhanced_result.model_dump_json(indent=2))


{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "In an increasingly dynamic professional landscape, Drucker's framework on self-management underscores the critical need for AI professionals to cultivate introspection, recognize their strengths, and align their values with organizational goals. This proactive approach is essential for thriving in a knowledge economy where traditional career paths are rapidly evolving, thus highlighting the relevance of self-managing one's career in the context of both personal fulfillment and professional success.",
  "Summary": "In his seminal work 'Managing Oneself,' Peter F. Drucker delineates the imperative for individuals in the contemporary knowledge economy to proactively assume control over their career trajectories, akin to the role of a Chief Executive Officer. Recognizing the diminished role of organizations in career management, Drucker emphasizes that individuals must develop self-awareness encompassing their

In [13]:
enhanced_eval = evaluate_summary(
    document_text=document_text,
    summary_text=enhanced_result.Summary,
    tone=tone
)

print(json.dumps(enhanced_eval, indent=2))


{
  "SummarizationScore": 0.9,
  "SummarizationReason": "The summary effectively captures the core ideas of Drucker's work, including the importance of self-awareness, leveraging strengths, and the need for ethical alignment with organizational values. It covers the main themes and provides a comprehensive overview of the article's content.",
  "CoherenceScore": 0.85,
  "CoherenceReason": "The summary is generally clear and logically structured, presenting the ideas in a coherent manner. However, some sentences could be more concise to enhance clarity and reduce redundancy.",
  "TonalityScore": 0.4,
  "TonalityReason": "The summary maintains a neutral and analytic tone but does not align well with the bureaucratese target tone, which typically features more formal and complex language. The language used is straightforward and lacks the bureaucratic style expected.",
  "SafetyScore": 1.0,
  "SafetyReason": "The content of the summary is safe, containing no harmful, unsafe, discriminator

In [15]:
print("========== COMPARISON ==========")
print("Original SummarizationScore:", evaluation_result["SummarizationScore"])
print("Enhanced SummarizationScore:", enhanced_eval["SummarizationScore"])

print("Original CoherenceScore:", evaluation_result["CoherenceScore"])
print("Enhanced CoherenceScore:", enhanced_eval["CoherenceScore"])

print("Original TonalityScore:", evaluation_result["TonalityScore"])
print("Enhanced TonalityScore:", enhanced_eval["TonalityScore"])

print("Original SafetyScore:", evaluation_result["SafetyScore"])
print("Enhanced SafetyScore:", enhanced_eval["SafetyScore"])


========== COMPARISON ==========
Original SummarizationScore: 0.9
Enhanced SummarizationScore: 0.9
Original CoherenceScore: 0.85
Enhanced CoherenceScore: 0.85
Original TonalityScore: 0.4
Enhanced TonalityScore: 0.4
Original SafetyScore: 1.0
Enhanced SafetyScore: 1.0


# Comments

I used gpt-4o-mini via the API Gateway.

I generated a structured output using Pydantic.

DeepEval could not run in my environment due to dependency conflicts (langchain + pydantic), so I implemented an LLM-as-a-judge evaluation function that reproduces the same metric categories and uses bespoke assessment criteria.

After enhancement, the summary quality stayed similar in score. This suggests the original summary already satisfied most evaluation constraints, and the enhancement primarily improved style consistency rather than coverage.

These controls are helpful but not sufficient alone; evaluation is still subjective and should ideally include multiple judges, deterministic checks, and/or reference-based scoring.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
